In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Subset, DataLoader
from sklearn.model_selection import StratifiedKFold
import numpy as np


from dataset import create_dataloaders
from tcn_model import MEGTCN
from gan_model import MEGGAN
from train import train_one_epoch
from evaluate import evaluate
from grid_search import run_grid_search

In [2]:
DATA_DIR = "preprocessed_data/Intra"
BATCH_SIZE = 8
EPOCHS = 30
LEARNING_RATE = 1e-3
NUM_CLASSES = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
train_loader, test_loader = create_dataloaders(DATA_DIR, BATCH_SIZE)

Loading test data...
Loading train data...
Loaded 32 training samples
Loaded 8 test samples
Class distribution in training: [8 8 8 8]
Class distribution in test: [2 2 2 2]
Train batches: 4
Test batches: 1


In [4]:

# K-fold cross-validation with stratification
dataset = train_loader.dataset
k_folds = 4

# Get labels from dataset
labels = np.array([dataset[i][1] for i in range(len(dataset))])

# Stratified k-fold split
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(dataset)), labels)):
    fold_train_loader = DataLoader(Subset(dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(Subset(dataset, val_idx), batch_size=BATCH_SIZE, shuffle=False)

    model = MEGGAN(num_classes=NUM_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    print(f"Starting fold {fold+1}/{k_folds}")
    for epoch in range(EPOCHS):
        train_loss, train_acc = train_one_epoch(model, fold_train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)

        print(
            f"Fold {fold+1} | Epoch {epoch+1}/{EPOCHS} | "
            f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%"
        )


Starting fold 1/4
Fold 1 | Epoch 1/30 | Train Acc: 16.67% | Val Acc: 25.00%
Fold 1 | Epoch 2/30 | Train Acc: 41.67% | Val Acc: 25.00%
Fold 1 | Epoch 3/30 | Train Acc: 58.33% | Val Acc: 25.00%
Fold 1 | Epoch 4/30 | Train Acc: 45.83% | Val Acc: 25.00%
Fold 1 | Epoch 5/30 | Train Acc: 50.00% | Val Acc: 25.00%
Fold 1 | Epoch 6/30 | Train Acc: 75.00% | Val Acc: 25.00%
Fold 1 | Epoch 7/30 | Train Acc: 83.33% | Val Acc: 25.00%
Fold 1 | Epoch 8/30 | Train Acc: 79.17% | Val Acc: 25.00%
Fold 1 | Epoch 9/30 | Train Acc: 58.33% | Val Acc: 25.00%
Fold 1 | Epoch 10/30 | Train Acc: 75.00% | Val Acc: 50.00%
Fold 1 | Epoch 11/30 | Train Acc: 66.67% | Val Acc: 37.50%
Fold 1 | Epoch 12/30 | Train Acc: 75.00% | Val Acc: 50.00%
Fold 1 | Epoch 13/30 | Train Acc: 83.33% | Val Acc: 50.00%
Fold 1 | Epoch 14/30 | Train Acc: 87.50% | Val Acc: 62.50%
Fold 1 | Epoch 15/30 | Train Acc: 79.17% | Val Acc: 75.00%
Fold 1 | Epoch 16/30 | Train Acc: 95.83% | Val Acc: 62.50%
Fold 1 | Epoch 17/30 | Train Acc: 87.50% | Val 

In [5]:
param_grid = {
    "learning_rate": [1e-3, 5e-4],
    "kernel_size": [3, 5],
    "dropout": [0.2],
    "hidden_channels": [32],
    "batch_size": [16],
}

top_models = run_grid_search(
    model_class=MEGTCN,
    param_grid=param_grid,
    train_loader=train_loader,
    num_classes=4,
    epochs=20,
)


Testing parameters:
{'learning_rate': 0.001, 'kernel_size': 3, 'dropout': 0.2, 'hidden_channels': 32, 'batch_size': 16}
Starting fold 1/4
Fold 1 | Epoch 1/20 | Train Acc: 33.33% | Val Acc: 25.00%
Fold 1 | Epoch 2/20 | Train Acc: 83.33% | Val Acc: 25.00%
Fold 1 | Epoch 3/20 | Train Acc: 100.00% | Val Acc: 87.50%
Fold 1 | Epoch 4/20 | Train Acc: 91.67% | Val Acc: 100.00%
Fold 1 | Epoch 5/20 | Train Acc: 100.00% | Val Acc: 100.00%
Fold 1 | Epoch 6/20 | Train Acc: 100.00% | Val Acc: 100.00%
Fold 1 | Epoch 7/20 | Train Acc: 100.00% | Val Acc: 100.00%
Fold 1 | Epoch 8/20 | Train Acc: 100.00% | Val Acc: 100.00%
Fold 1 | Epoch 9/20 | Train Acc: 100.00% | Val Acc: 100.00%
Fold 1 | Epoch 10/20 | Train Acc: 100.00% | Val Acc: 100.00%
Fold 1 | Epoch 11/20 | Train Acc: 100.00% | Val Acc: 100.00%
Fold 1 | Epoch 12/20 | Train Acc: 100.00% | Val Acc: 100.00%
Fold 1 | Epoch 13/20 | Train Acc: 95.83% | Val Acc: 100.00%
Fold 1 | Epoch 14/20 | Train Acc: 100.00% | Val Acc: 100.00%
Fold 1 | Epoch 15/20 | 